# Agentic Pipeline — Auto-Generated Notebook
**Pipeline ID:** `776c2d8e`  
**Status:** SUCCESS  
**Total time:** 80.21s


## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
import pandas as pd, warnings, sys
import warnings
_PIPELINE_ID = '776c2d8e'


## Step 1: Select And Train Models

**LLM Reasoning:** Agent     : DynamicAgent[select_and_train_models] Status    : SUCCESS Duration  : 9.579s Timestamp : 2026-04-25T11:37:49.559108+00:00 Input     : DataFrame(0x0) nulls=0.0


In [ ]:
# Verify df is a DataFrame
assert isinstance(df, pd.DataFrame), "df must be a pandas DataFrame"
# Initialize container for trained models
trained_models = {}
# Check for usable data
if df.empty or df.shape[1] == 0:
    print("DataFrame is empty (no rows or columns). Skipping model training.")
else:
    # Placeholder for actual training logic when data becomes available
    # Example: split, select algorithms, fit, store in trained_models
    # This block is unreachable with the current empty schema.
    pass
# Ensure df remains a DataFrame at the end
assert isinstance(df, pd.DataFrame), "df should still be a pandas DataFrame after this step"


## Step 2: Evaluate Models

**LLM Reasoning:** Agent     : DynamicAgent[evaluate_models] Status    : SUCCESS Duration  : 16.253s Timestamp : 2026-04-25T11:38:05.829501+00:00 Input     : ExecutionResult(df=0x0, model=NoneType)


In [ ]:
# Verify df is a DataFrame
assert isinstance(df, pd.DataFrame)
# Container for evaluation metrics
evaluation_results = {}
# Determine where trained models are stored
models = {}
if 'trained_models' in globals():
    models = trained_models  # dict of name -> fitted model
elif 'trained_model' in globals():
    models = {'model': trained_model}
else:
    print('No trained model(s) found in the environment.')
# If the DataFrame is empty or lacks a target column, skip evaluation
if df.empty or 'target' not in df.columns:
    print('Evaluation skipped: DataFrame is empty or missing a "target" column.')
else:
    X = df.drop(columns=['target'])
    y_true = df['target']
    for name, model in models.items():
        try:
            y_pred = model.predict(X)
        except Exception as e:
            print(f'Prediction failed for model {name}: {e}')
            continue
        # Compute a few generic metrics; if they fail (e.g., wrong problem type) store None
        metrics = {}
        try:
            metrics['accuracy'] = accuracy_score(y_true, y_pred)
        except Exception:
            metrics['accuracy'] = None
        try:
            metrics['f1'] = f1_score(y_true, y_pred, average='weighted')
        except Exception:
            metrics['f1'] = None
        try:
            metrics['mse'] = mean_squared_error(y_true, y_pred)
        except Exception:
            metrics['mse'] = None
        try:
            metrics['r2'] = r2_score(y_true, y_pred)
        except Exception:
            metrics['r2'] = None
        evaluation_results[name] = metrics
# Validation: ensure evaluation_results is a dict (empty if skipped)
assert isinstance(evaluation_results, dict), 'evaluation_results should be a dictionary'


## Step 3: Select Best Model

**LLM Reasoning:** Agent     : DynamicAgent[select_best_model] Status    : SUCCESS Duration  : 15.991s Timestamp : 2026-04-25T11:38:21.837460+00:00 Input     : ExecutionResult(df=0x0, model=NoneType)


In [ ]:
# Ensure df is still a DataFrame (pipeline rule)
assert isinstance(df, pd.DataFrame), "df must be a pandas DataFrame"
# Initialize placeholders
best_model_name = None
best_model = None
# Helper to log messages without breaking the pipeline
def _log(msg):
    print(msg, file=sys.stderr)
# 1. Prefer an explicit performance dict created by earlier evaluation step
if 'model_performance' in globals() and isinstance(model_performance, dict) and model_performance:
    # Assume higher metric is better (e.g., accuracy, f1). If lower is better, user should adjust.
    best_model_name = max(model_performance, key=model_performance.get)
    _log(f"Selected best model based on model_performance dict: {best_model_name} (score={model_performance[best_model_name]})")
elif 'trained_models' in globals() and isinstance(trained_models, dict) and trained_models:
    # Try to infer a score attribute from each model (common in sklearn meta‑estimators)
    scores = {}
    for name, mdl in trained_models.items():
        # Look for typical attributes that store a validation score
        for attr in ['best_score_', 'score_', 'validation_score']:
            if hasattr(mdl, attr):
                scores[name] = getattr(mdl, attr)
                break
    if scores:
        best_model_name = max(scores, key=scores.get)
        _log(f"Selected best model based on inferred score attribute: {best_model_name} (score={scores[best_model_name]})")
    else:
        _log("No explicit performance dict or score attributes found in trained_models; cannot determine best model.")
else:
    _log("No trained_models or model_performance objects found in the environment.")
# Retrieve the actual model object if a name was identified
if best_model_name and 'trained_models' in globals():
    best_model = trained_models.get(best_model_name)
    # Store for downstream steps
    globals()['best_model'] = best_model
    globals()['best_model_name'] = best_model_name
else:
    globals()['best_model'] = None
    globals()['best_model_name'] = None
    _log("best_model set to None due to lack of evaluation data.")
# Validation: report what was selected
print(f"BEST_MODEL_NAME: {best_model_name}")
print(f"BEST_MODEL object: {best_model}")


## Step 4: Explain Model

**LLM Reasoning:** Agent     : DynamicAgent[explain_model] Status    : SUCCESS Duration  : 14.837s Timestamp : 2026-04-25T11:38:36.690829+00:00 Input     : ExecutionResult(df=0x0, model=NoneType)


In [ ]:
# Ensure df is still a DataFrame
assert isinstance(df, pd.DataFrame), "df must be a pandas DataFrame"
explanation = None
# Helper to attempt SHAP explanation if possible
def _explain_with_shap(model, background):
    try:
        # Use KernelExplainer for generic models
        explainer = shap.KernelExplainer(model.predict, background)
        # Explain a single synthetic instance (zeros) matching background shape
        shap_values = explainer.shap_values(background[:1])
        return shap_values
    except Exception as e:
        warnings.warn(f"SHAP explanation failed: {e}")
        return None
# Determine if a model is available
model = None
if 'trained_model' in globals():
    model = trained_model
elif 'trained_models' in globals() and isinstance(trained_models, dict) and len(trained_models) > 0:
    # Pick the first model in the dict for explanation
    model_name, model = next(iter(trained_models.items()))
    print(f"Using model '{model_name}' for explanation.")
else:
    print("No trained model found in the environment; skipping explanation step.")
if model is not None:
    # Try to infer expected input shape from the model if possible
    # Since df has no columns, create a dummy background of zeros with a generic size
    # Attempt to get number of features via model.n_features_in_ if attribute exists
    n_features = getattr(model, 'n_features_in_', None)
    if n_features is None:
        # Fallback: try to inspect the first layer of sklearn linear models
        if hasattr(model, 'coef_'):
            n_features = model.coef_.shape[-1]
    if n_features is None:
        print("Unable to infer number of features from the model; cannot create dummy background for SHAP.")
    else:
        # Create dummy background with 10 samples of zeros
        dummy_background = np.zeros((10, n_features))
        explanation = _explain_with_shap(model, dummy_background)
        if explanation is not None:
            print("SHAP explanation computed successfully.")
        else:
            print("SHAP explanation could not be computed.")
else:
    explanation = None
# Validation: ensure explanation variable exists (even if None)
assert 'explanation' in globals(), "Explanation variable was not created"
# SAVE OUTPUTS
import os
os.makedirs('outputs', exist_ok=True)
# Save cleaned dataset
df.to_csv('outputs/cleaned_data.csv', index=False, encoding='utf-8')
print(f'Cleaned data saved -> outputs/cleaned_data.csv ({df.shape[0]} rows x {df.shape[1]} cols)')
# Save trained model
import joblib
if 'trained_model' in dir():
    joblib.dump(trained_model, 'outputs/model.pkl')
    print(f'Model saved -> outputs/model.pkl ({type(trained_model).__name__})')
elif 'trained_models' in dir() and isinstance(trained_models, dict):
    # Save best model (first in dict) and all models
    best_name, best_model = next(iter(trained_models.items()))
    joblib.dump(best_model, 'outputs/model.pkl')
    print(f'Best model saved -> outputs/model.pkl ({best_name}: {type(best_model).__name__})')
    # Save all models
    for name, model in trained_models.items():
        joblib.dump(model, f'outputs/model_{name}.pkl')
    print(f'All {len(trained_models)} models saved to outputs/')
else:
    print('WARNING: No trained model found to save')
print('All outputs saved to outputs/ directory')


## Saved Outputs

In [ ]:
# ── Load pipeline outputs ────────────────────────────
import pandas as pd

# Cleaned dataset
df_clean = pd.read_csv('outputs/cleaned_data.csv', encoding='utf-8')
print(f'Cleaned data: {df_clean.shape[0]} rows x {df_clean.shape[1]} cols')
df_clean.head()
